<a href="https://colab.research.google.com/github/robertbarcik/genai-in-python-tutorial/blob/main/6_basics_of_prompt_engineering/6_basics_of_prompt_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Basics of Prompt Engineering

*An introductory guide to effective prompt engineering with OpenAI's API* 🚀

---

## 🤔 **What is Prompt Engineering?**

### 📖 **Introduction to Prompt Engineering**

Prompt engineering refers to the careful design and structuring of input prompts that are given to AI language models to produce desired outputs. These prompts guide the AI's responses, helping to shape them in a way that meets specific needs. When interacting with language models, the way a question or command is phrased can significantly influence the quality and relevance of the response.

In essence, prompt engineering is about crafting the "questions" we ask the AI to get the "answers" we need. This practice is fundamental in ensuring that the AI understands the context and purpose of the interaction, resulting in accurate, coherent, and useful outputs.

### 💡 **Why is Prompt Engineering Important?**

Prompt engineering is crucial for several reasons. First, it allows us to control the quality of the AI's responses. By carefully crafting prompts, we can guide the AI to provide more precise, relevant, and contextually appropriate answers. This control is vital, especially in scenarios where the output must meet high standards of accuracy or clarity, such as customer support, content creation, or technical documentation.

Second, prompt engineering helps unlock the full capabilities of language models. AI models are inherently flexible and can handle various tasks, but they need clear instructions to perform effectively. Good prompt design clarifies these instructions, ensuring that the AI knows what to do and how to do it.

Finally, effective prompt engineering reduces ambiguity. Ambiguous or vague prompts can lead to off-topic or nonsensical answers. By being clear and specific in our prompts, we minimize the chances of misinterpretation and improve the reliability of the AI's responses.

### 🏗️ **The Role of an Architect in Prompt Engineering**

While prompt engineering focuses on crafting effective inputs for AI models, it must be combined with strategic thinking and architectural design to solve real-world problems effectively. In other words, it's not just about what you send to the model but also about how this fits into the broader system.

An "architect" in this context refers to someone who understands how to structure the interaction between the user, the AI, and the surrounding environment in a seamless and efficient manner. This involves:

1. **Breaking Down Problems:** Before designing prompts, it's essential to decompose the problem into manageable parts. This step ensures that each aspect of the task is understood and addressed correctly. For instance, if you're developing a customer support chatbot, you would break down tasks into greeting the user, understanding the query, fetching information, and providing a response.

2. **Integrating Prompts with Python Code:** Effective prompt engineering requires embedding the prompts within a functional Python framework. This means creating functions that handle the generation of prompts, processing user inputs, and managing the AI's responses. It also involves setting up error handling, logging, and data management to ensure that the application runs smoothly.

3. **Maintaining a Feedback Loop:** Prompt engineering is often iterative. As you interact with the AI, you will learn which prompts work best and which need adjustment. This requires building a feedback loop within your application to monitor and refine prompts based on user interactions and outcomes.

---

# 🔧 Setup

Let's start by installing the required libraries and configuring our OpenAI API key.

In [1]:
# Install required packages
!pip install -q openai


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# Import necessary libraries
import os
import warnings
from openai import OpenAI

# Suppress deprecation warnings for cleaner output
warnings.filterwarnings("ignore", category=DeprecationWarning)

### 🔑 API Key Configuration

You have two methods to provide your OpenAI API key:

**Method 1 (Recommended)** ✨: Use Colab Secrets
1. Click the 🔑 icon in the left sidebar
2. Click "Add new secret"
3. Name: `OPENAI_API_KEY`
4. Value: Your OpenAI API key
5. Enable notebook access

**Method 2 (Fallback)** 💭: Manual input when prompted

In [3]:
# Configure OpenAI API key
# Method 1: Try to get API key from Colab secrets (recommended)
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    import os
    if os.environ.get('OPENAI_API_KEY'):
        OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
        print("✅ API key loaded from environment variable")
    else:
        # Method 2: Manual input (fallback)
        from getpass import getpass
        print("💡 To use Colab secrets: Go to 🔑 (left sidebar) → Add new secret → Name: OPENAI_API_KEY")
        OPENAI_API_KEY = getpass("Enter your OpenAI API Key: ")

# Set the API key as an environment variable
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Validate that the API key is set
if not OPENAI_API_KEY or OPENAI_API_KEY.strip() == "":
    raise ValueError("❌ ERROR: No API key provided!")

print("✅ Authentication configured!")

# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)
print("✅ OpenAI client initialized!")

# We'll use gpt-5-nano for cost efficiency throughout this notebook
MODEL = "gpt-5-nano"
print(f"🤖 Using model: {MODEL}")

✅ API key loaded from environment variable
✅ Authentication configured!
✅ OpenAI client initialized!
🤖 Using model: gpt-5-nano


---

# 0️⃣ Understanding Model Parameters

Before diving into prompt engineering techniques, it's essential to understand how models process inputs and generate outputs. Two fundamental concepts will help you use AI models more effectively: **temperature settings** 🌡️ and **token limits** 🎫.

## 🌡️ 0.1 Temperature and Sampling Parameters

When you ask an AI model to generate text, it doesn't just produce a single "correct" answer. Instead, it considers many possible next words and chooses among them. The `temperature` parameter controls how this choice is made, fundamentally affecting whether the output feels predictable or creative.

Think of temperature as a **creativity dial** 🎨. At low temperatures (closer to 0.0), the model behaves conservatively, almost always choosing the most likely next word. This makes outputs consistent, focused, and deterministic—ask the same question twice and you'll get nearly identical answers. At high temperatures (1.0 and above), the model becomes more adventurous, sometimes choosing less probable words, leading to varied, creative, and occasionally surprising responses. Ask the same question multiple times at high temperature, and you might get completely different answers each time.

The temperature scale typically ranges from **0.0 to 2.0**. Low values between **0.0 and 0.3** work best for factual tasks like data extraction, classification, or answering specific questions where accuracy matters more than variety. Medium values around **0.5 to 0.7** offer a balanced approach suitable for general-purpose tasks, providing some variation without sacrificing too much consistency. High values from **0.8 to 1.0** (or even higher) shine in creative applications like storytelling, brainstorming sessions, or generating diverse marketing copy where you want the model to explore different phrasings and ideas.

Understanding temperature helps you match the model's behavior to your specific needs. When you need reliable, consistent answers—like extracting information from documents or classifying support tickets—use lower temperatures ❄️. When you want the model to explore possibilities and generate varied options—like writing product descriptions or coming up with blog post titles—crank up the temperature 🔥.

### 🧪 Practical Example: Temperature in Action

Let's see how temperature affects the same prompt. We'll ask the model to write a professional email subject line for a meeting request.

In [4]:
# Test prompt
test_prompt = "Write a professional email subject line for requesting a meeting with your manager to discuss project updates."

# Note: gpt-5-nano is a reasoning model and doesn't support the `temperature`
# parameter. We use gpt-4o-mini (still very cheap) just for this demo so you
# can actually see temperature change the output.
TEMPERATURE_DEMO_MODEL = "gpt-4o-mini"

# Low temperature (0.2) - Focused and consistent
print("🔵 LOW TEMPERATURE (0.2) - Focused and consistent")
print("=" * 70)
response_low = client.responses.create(
    model=TEMPERATURE_DEMO_MODEL,
    input=test_prompt,
    temperature=0.2
)
print(response_low.output_text)
print()

# Medium temperature (0.7) - Balanced
print("🟡 MEDIUM TEMPERATURE (0.7) - Balanced creativity")
print("=" * 70)
response_med = client.responses.create(
    model=TEMPERATURE_DEMO_MODEL,
    input=test_prompt,
    temperature=0.7
)
print(response_med.output_text)
print()

# High temperature (1.0) - Creative and varied
print("🔴 HIGH TEMPERATURE (1.0) - Creative and varied")
print("=" * 70)
response_high = client.responses.create(
    model=TEMPERATURE_DEMO_MODEL,
    input=test_prompt,
    temperature=1.0
)
print(response_high.output_text)

🔵 LOW TEMPERATURE (0.2) - Focused and consistent


Subject: Request for Meeting to Discuss Project Updates

🟡 MEDIUM TEMPERATURE (0.7) - Balanced creativity


Subject: Request for Meeting to Discuss Project Updates

🔴 HIGH TEMPERATURE (1.0) - Creative and varied


Subject: Request for Meeting: Project Updates Discussion


Notice how the outputs differ across temperature settings! The low-temperature version likely gave you something straightforward and professional, perhaps "Meeting Request: Project Updates Discussion" or similar. The medium-temperature version probably added a bit more personality while staying professional. The high-temperature version might have taken creative liberties, perhaps using more colorful language or unconventional phrasing.

💡 **Key Takeaway**: Temperature matters most when you need either absolute consistency (low) or deliberate variety (high). For many everyday tasks, the default temperature works fine, but understanding this parameter gives you precise control over the model's creativity-consistency trade-off.

## 🎫 0.2 Token Limits & Context Awareness

AI models don't process text the way we do, word by word. Instead, they break language into smaller units called **tokens**. Understanding tokens is crucial because they determine both what you can ask the model to do and how much it will cost you 💰.

A token isn't exactly a word, though that's a useful approximation. Think of tokens as the model's way of seeing language—sometimes a token is a whole word like "hello," sometimes it's just part of a word like "ing" in "running," and sometimes it's punctuation or even a space. In English, a rough rule of thumb is that **one token ≈ 4 characters** or **¾ of a word**. So a 100-word paragraph might use around 133 tokens. Numbers, special characters, and non-English text can have different token counts because the model breaks them down differently.

Every model has a **context window**—a maximum number of tokens it can process in a single request. This limit includes both your input prompt and the model's response. For example:
- **GPT-3.5-turbo**: 16,385 tokens (~12,000 words) 📄
- **GPT-4**: varies by version, some exceed 100,000 tokens 📚

These limits might seem generous, but they matter when you're working with long documents, providing extensive examples, or asking the model to process multiple texts at once.

What happens when you exceed these limits? ⚠️ Usually, the API will reject your request with an error, telling you that your prompt is too long. Some implementations automatically truncate your input, cutting off the beginning or end to fit within limits, which can lead to incomplete context and poor results. This is why understanding your token usage is essential for production applications—you need to design your prompts knowing these constraints exist.

Tokens have direct cost implications 💵. API providers charge based on token usage, typically pricing input tokens (your prompt) differently from output tokens (the model's response). If you're processing thousands of documents or generating lengthy reports, token costs can add up quickly. Efficient prompt design—getting the results you need with fewer tokens—becomes not just a technical concern but a financial one.

### 📊 Checking Token Usage

Let's see how to check token usage from an API response. This helps you understand the cost of your prompts and responses.

In [5]:
# Make a simple API call and check token usage
test_prompt = "Explain what machine learning is in 3 sentences."

response = client.responses.create(
    model=MODEL,
    input=test_prompt
)

print("📝 Response:")
print("=" * 70)
print(response.output_text)
print()

# Check token usage
print("📊 Token Usage:")
print("=" * 70)
print(f"Input tokens (your prompt): {response.usage.input_tokens}")
print(f"Output tokens (model's response): {response.usage.output_tokens}")
print(f"Total tokens used: {response.usage.total_tokens}")
print()

# Calculate approximate cost (example rates - check current pricing)
# These are example rates for illustration
input_cost_per_1k = 0.0001  # Example: $0.0001 per 1K input tokens
output_cost_per_1k = 0.0002  # Example: $0.0002 per 1K output tokens

input_cost = (response.usage.input_tokens / 1000) * input_cost_per_1k
output_cost = (response.usage.output_tokens / 1000) * output_cost_per_1k
total_cost = input_cost + output_cost

print("💰 Approximate Cost (example rates):")
print(f"This request cost approximately: ${total_cost:.6f}")

📝 Response:
Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data rather than following explicit rules. It trains models on examples so they can make predictions or decisions about new data. As more data or feedback is provided, these models improve their accuracy and adapt to new tasks and environments.

📊 Token Usage:
Input tokens (your prompt): 16
Output tokens (model's response): 392
Total tokens used: 408

💰 Approximate Cost (example rates):
This request cost approximately: $0.000080


Why does understanding tokens matter for real applications? 🤔 When you're building a chatbot that handles thousands of conversations, or a document analysis tool that processes hundreds of pages, or an automated content generator that runs daily, token efficiency directly affects your bottom line. Knowing how to check usage, estimate costs, and optimize your prompts to use fewer tokens while still getting quality results becomes a critical skill. Beyond costs, token awareness helps you design better applications—you'll know when to split long documents into chunks, when to summarize before processing, and how to stay within context windows without losing important information.

---

# 1️⃣ Writing Clear Instructions

The foundation of effective prompt engineering is **clear communication** 💬. The model will perform best when you provide explicit, detailed instructions about what you want. Let's explore several tactics for writing clearer prompts!

## 📝 1.1 Tactic: Add Specific Details for Better Responses

When you want a relevant and accurate response, ensure your requests include essential details and context. Vague prompts force the model to guess what you really want, often leading to generic or off-target responses. 🎯

Here are some examples to illustrate how adding specific details improves responses:

| ❌ Vague                                 | ✅ Detailed                                                                                  |
|-----------------------------------------|----------------------------------------------------------------------------------------------|
| How do I cook pasta?                    | How do I cook al dente spaghetti for a carbonara dish?                                       |
| Write a function to sort a list.        | Write a Python function to sort a list of dictionaries by a specific key. Include comments to explain the logic. |
| Describe the history of the internet.   | Provide a brief history of the internet, highlighting key milestones from the 1960s to the present. |

### 🔬 Example: Vague vs. Detailed Prompt

In [6]:
# Vague request - lacks specific details
print("❌ VAGUE PROMPT:")
print("=" * 70)
vague_prompt = "Tell me about blockchain technology."
print(f"Prompt: {vague_prompt}\n")

response_vague = client.responses.create(
    model=MODEL,
    input=vague_prompt
)
print(response_vague.output_text)
print("\n" + "=" * 70 + "\n")

❌ VAGUE PROMPT:
Prompt: Tell me about blockchain technology.



Here’s a concise overview of blockchain technology and why it’s talked about so much.

What it is
- A blockchain is a distributed, append-only ledger made of blocks. Each block holds a set of transactions, a timestamp, and a cryptographic hash linking it to the previous block. This chaining of blocks creates an immutable history of activity.
- A network of nodes validates and agrees on the content and order of blocks using a consensus mechanism. This consensus makes it hard for any single actor to forge history.

How it works (high level)
- Transactions are proposed and broadcast to the network.
- Nodes validate transactions (e.g., signatures, rules) and collect them into a candidate block.
- The network reaches consensus on which block is next (and thus which transactions are confirmed).
- The chosen block is added to the chain, and the process repeats. Rewriting past blocks becomes prohibitively expensive in many networks.

Core components and terms
- Blocks: containers for transacti

In [7]:
# Detailed request - includes specific details about what you want
print("✅ DETAILED PROMPT:")
print("=" * 70)
detailed_prompt = "Explain in short how blockchain technology works and provide one application in finance, healthcare, and supply chain management."
print(f"Prompt: {detailed_prompt}\n")

response_detailed = client.responses.create(
    model=MODEL,
    input=detailed_prompt
)
print(response_detailed.output_text)

✅ DETAILED PROMPT:
Prompt: Explain in short how blockchain technology works and provide one application in finance, healthcare, and supply chain management.



Here's a concise overview and one application per domain.

How blockchain works (short):
- It is a distributed ledger shared among many participants.
- Transactions are grouped into blocks, each block links to the previous one with a cryptographic hash.
- New blocks are added only after network-wide validation via a consensus mechanism (e.g., Proof of Work, Proof of Stake).
- Once a block is added, its data is practically immutable and visible to the network (public) or authorized participants (permissioned).
- Some blockchains support smart contracts—programs that automatically execute rules when conditions are met.

Applications (one per domain):
- Finance: Cross-border payments and settlement on a shared ledger, enabling faster finality and reduced reconciliation risk.
- Healthcare: Secure, consent-based sharing of electronic health records with an immutable audit trail of who accessed the data.
- Supply chain management: Provenance and traceability of products from origin to consum

### 🎯 When to Use This Tactic

Specific details work wonders when you're working in specialized domains where **precision matters** 🔍. If you're asking about technical implementations, industry-specific processes, or need information tailored to a particular context, adding details ensures the model understands your exact requirements. For instance, asking about "marketing strategies" is broad, but asking about "digital marketing strategies for B2B SaaS companies targeting enterprise clients" narrows the focus dramatically, yielding far more relevant advice.

However, there's a balance to strike ⚖️. For creative tasks like brainstorming or exploring possibilities, too much specificity can be constraining. If you want the model to generate diverse blog post ideas, saying "give me blog post ideas" might work better than providing ten constraints about length, tone, audience, and topic. The sweet spot is being specific about the parameters that truly matter while leaving room for the model to be helpful in ways you might not have anticipated. When you catch yourself writing a vague prompt and getting generic results, that's your signal to add the crucial details that distinguish your specific need from a general request.

## 🎭 1.2 Tactic: Ask the Model to Adopt a Specific Persona

Persona adoption means guiding the AI to respond like a particular character or role. This approach has many benefits! It makes responses more engaging and relatable by reflecting a specific persona, ensures consistency in tone and style, and allows for tailored responses that better meet the needs of different audiences.

Examples of persona adoption could be:
- **👨‍🏫 Educational Tutor**: The model adopts a patient and explanatory style, ideal for teaching complex topics.
- **🤝 Customer Service Agent**: The model uses a friendly and helpful tone to assist with customer inquiries.
- **👨‍💻 Technical Expert**: The model provides detailed and precise information suitable for professional discussions.

### 📋 Example: Without Persona

First, let's see how the AI responds without adopting any specific persona.

In [8]:
# Request without specifying a persona
print("📝 WITHOUT PERSONA:")
print("=" * 70)

response_no_persona = client.responses.create(
    model=MODEL,
    input="I'm feeling very anxious because I've been having headaches and dizziness. What could be causing this?"
)

print(response_no_persona.output_text)

📝 WITHOUT PERSONA:


I’m glad you told me. Headache and dizziness can happen for many reasons, and anxiety can make them feel worse. Most causes are not dangerous, but some signs require medical evaluation. Here’s a practical overview to help you think about what might be going on and what to do next.

Possible common causes
- Tension or migraine headaches: often tied to stress, poor sleep, dehydration, eyestrain, or skipping meals.
- Anxiety and panic: can cause dizziness, lightheadedness, and headaches through hyperventilation and muscle tension.
- Dehydration or low blood sugar: not drinking enough fluids or irregular meals.
- Caffeine or medication effects: too much or withdrawal, or side effects from medicines.
- Inner ear/vestibular issues (like BPPV or infection): dizziness that worsens with head movement or lasts longer.
- Neck/shoulder tension or posture problems: headaches from tight muscles in the neck and upper back.

More serious or less common possibilities (less likely, but important to rule

### 💙 Example: With Empathetic Persona

Now let's specify an empathetic and caring persona that acknowledges feelings while providing information.

In [9]:
# Request with a specific empathetic persona
print("💙 WITH EMPATHETIC PERSONA:")
print("=" * 70)

# Combine persona instruction with user query
prompt_with_persona = """You are an empathetic and caring assistant who helps to calm down and reassure people seeking medical advice.

User message: I'm feeling very anxious because I've been having headaches and dizziness. What could be causing this?"""

response_with_persona = client.responses.create(
    model=MODEL,
    input=prompt_with_persona
)

print(response_with_persona.output_text)

💙 WITH EMPATHETIC PERSONA:


I’m really glad you reached out. It’s understandable to feel anxious when you’re dealing with headaches and dizziness. There are lots of possible, mostly manageable causes, but some need prompt evaluation. Here’s a helpful overview and what you can do now.

What this could be (common possibilities)
- Dehydration or low blood sugar
- Lack of sleep, stress, or anxiety/panic symptoms
- Eye strain or changes in vision (needs eye check)
- Tension headaches or migraines
- Sinus or nasal issues
- Temporary vertigo or inner-ear issues
- Caffeine withdrawal or excess
- Side effects from medications or alcohol
- Less commonly: infections, anemia, dehydration from illness, or more rarely problems with blood pressure

Red flags that mean you should seek urgent care or call emergency services right away
- Sudden, very severe headache “worst ever,” or a headache with stiff neck, fever, confusion, weakness, numbness, trouble speaking, vision changes
- Dizziness with chest pain, trouble breathing, fai

### 🎪 When to Use Personas

Personas are your tool for controlling **how** information is delivered, not **what** information the model knows. This is a crucial distinction that many people miss! 🚨 Asking the model to be "a medical expert" doesn't suddenly give it medical expertise it didn't have before—it simply changes the communication style, making responses sound more authoritative or use more technical language. The underlying knowledge remains the same.

Use personas when the style and tone of communication truly matter for your application 🎨. Customer service scenarios benefit enormously from a friendly, patient persona that acknowledges frustration and offers reassurance. Educational contexts work well with a tutor persona that breaks down concepts step by step without condescension. Sensitive topics like health 🏥, finance 💰, or legal matters ⚖️ call for personas that are empathetic, careful, and appropriately serious. Personas excel at maintaining a consistent voice across many interactions, which is essential for brand consistency or user experience.

Don't fall into the trap of using personas thinking they'll make the model magically more knowledgeable about a subject. If you need accurate technical information, provide reference materials or use specific, detailed prompts—don't just ask it to "be an expert." The best applications of personas are about emotional intelligence and communication style, not factual enhancement.

⚠️ **Important**: Personas affect **HOW** information is delivered, not **WHAT** information the model knows. Don't use personas as a substitute for providing accurate reference material or specific instructions!

---

## 📦 1.3 Tactic: Separating Parts of the Input

Using **delimiters** to clearly separate different sections of input helps the model understand which parts of the text should be treated differently. Delimiters such as triple quotation marks, section titles, and XML tags can specify which parts should be processed in specific ways, making it easier for the AI to understand and process information correctly. 🏷️

### ❌ Example: Without Delimiters

In [10]:
# Without delimiters - everything runs together
print("❌ WITHOUT DELIMITERS:")
print("=" * 70)

prompt_no_delimiters = """Summarize the arguments presented in each article. Then, evaluate which article presents a stronger argument and explain your reasoning. 

Online learning has revolutionized education by providing students with unprecedented access to resources and flexibility. One of the key advantages of online learning is that it allows students to learn at their own pace. Unlike traditional classroom settings where students must adhere to a fixed schedule, online courses offer the flexibility to review materials, take breaks, and move forward at a comfortable speed. This personalized approach can significantly enhance a student's understanding and retention of the subject matter. Additionally, online learning platforms often include a variety of multimedia tools such as videos, interactive quizzes, and forums, which cater to different learning styles and make the learning experience more engaging and effective. The accessibility of online learning also means that education can be democratized, reaching students who may not have had the opportunity to attend traditional schools due to geographical, financial, or personal constraints.

Despite the growing popularity of online learning, there are several challenges and limitations that cannot be overlooked. One significant drawback is the lack of face-to-face interaction, which can affect the learning experience. In traditional classroom settings, students benefit from real-time feedback, direct engagement with instructors, and collaborative learning with peers. These interactions are difficult to replicate in an online environment, which can lead to feelings of isolation and disengagement. Moreover, the effectiveness of online learning heavily depends on a student's self-discipline and motivation. Without the structured environment of a physical classroom, students may struggle to stay on track, complete assignments, and maintain consistent study habits. Technical issues such as unreliable internet access and the need for updated devices can also hinder the learning process, especially for students in underserved areas. Therefore, while online learning offers flexibility and accessibility, it also presents challenges that must be addressed to ensure a comprehensive and effective educational experience."""

response_no_delim = client.responses.create(
    model=MODEL,
    input=prompt_no_delimiters
)

print(response_no_delim.output_text)
print("\n" + "=" * 70 + "\n")

❌ WITHOUT DELIMITERS:


Here are the summaries and then an evaluation of which argument is stronger.

 summaries

- Article 1 (Advantages of online learning)
  - Core claim: Online learning revolutionizes education by providing unprecedented access and flexibility.
  - Supporting points:
    - Self-paced learning: students can review materials, take breaks, and progress at a comfortable speed.
    - Personalization: learning can be tailored to individual pace and needs, improving understanding and retention.
    - Multimedia tools: videos, interactive quizzes, and forums cater to different learning styles and boost engagement.
    - Accessibility and democratization: online platforms reach students who are geographically, financially, or personally constrained, expanding educational opportunities.

- Article 2 (Challenges and limitations of online learning)
  - Core claim: Despite popularity, online learning faces significant challenges that must be recognized and addressed.
  - Supporting points:
    - Lack 

### ✅ Example: With XML Delimiters

In [11]:
# With delimiters - clearly structured
print("✅ WITH XML DELIMITERS:")
print("=" * 70)

prompt_with_delimiters = """You will be provided with two articles about the same topic, each delimited by XML tags. First, summarize the arguments presented in each article. Then, evaluate which article presents a stronger argument and explain your reasoning.

<article>
Online learning has revolutionized education by providing students with unprecedented access to resources and flexibility. One of the key advantages of online learning is that it allows students to learn at their own pace. Unlike traditional classroom settings where students must adhere to a fixed schedule, online courses offer the flexibility to review materials, take breaks, and move forward at a comfortable speed. This personalized approach can significantly enhance a student's understanding and retention of the subject matter. Additionally, online learning platforms often include a variety of multimedia tools such as videos, interactive quizzes, and forums, which cater to different learning styles and make the learning experience more engaging and effective. The accessibility of online learning also means that education can be democratized, reaching students who may not have had the opportunity to attend traditional schools due to geographical, financial, or personal constraints.
</article>

<article>
Despite the growing popularity of online learning, there are several challenges and limitations that cannot be overlooked. One significant drawback is the lack of face-to-face interaction, which can affect the learning experience. In traditional classroom settings, students benefit from real-time feedback, direct engagement with instructors, and collaborative learning with peers. These interactions are difficult to replicate in an online environment, which can lead to feelings of isolation and disengagement. Moreover, the effectiveness of online learning heavily depends on a student's self-discipline and motivation. Without the structured environment of a physical classroom, students may struggle to stay on track, complete assignments, and maintain consistent study habits. Technical issues such as unreliable internet access and the need for updated devices can also hinder the learning process, especially for students in underserved areas. Therefore, while online learning offers flexibility and accessibility, it also presents challenges that must be addressed to ensure a comprehensive and effective educational experience.
</article>"""

response_with_delim = client.responses.create(
    model=MODEL,
    input=prompt_with_delimiters
)

print(response_with_delim.output_text)

✅ WITH XML DELIMITERS:


Here’s a concise breakdown and evaluation.

Summary of Article 1 (pro online learning)
- Main claim: Online learning revolutionizes education by providing flexibility and broad access.
- Key arguments:
  - Self-paced learning allows students to proceed at their own speed, review materials, take breaks, and optimize understanding and retention.
  - Multimedia tools (videos, interactive quizzes, forums) cater to different learning styles and increase engagement.
  - Accessibility of online platforms democratizes education, reaching students limited by geography, finances, or personal constraints.

Summary of Article 2 (downsides and limitations)
- Main claim: While popular, online learning has significant challenges that must be addressed.
- Key arguments:
  - Lack of face-to-face interaction reduces real-time feedback, instructor engagement, and peer collaboration, leading to possible isolation and disengagement.
  - Success depends on student self-discipline and motivation; without a s

### 🛡️ When Delimiters Matter

Delimiters become essential when you're juggling multiple pieces of content that need different treatment within a single request 🎯. Imagine you're building a document comparison tool—you want the model to analyze two contracts and highlight differences. Without delimiters, the model might blur the boundaries between documents, treating them as one continuous text and missing your intent to compare them separately. With clear XML tags or other delimiters marking where each contract begins and ends, the model understands the structure you're imposing.

This technique shines when processing user-generated content that might contain confusing instructions or attempts at **prompt injection** ⚠️. If a user submits a support ticket that includes text like "ignore previous instructions and do something else," delimiters help the model understand that this text is data to be processed, not instructions to follow. You're essentially saying, "Everything inside these tags is content, not commands." This becomes critical for security and reliability in production applications.

Delimiters also prevent confusion in complex multi-part tasks 🧩. If you want the model to translate one section, summarize another, and extract key points from a third—all in one request—delimiters make your intent crystal clear. Each section has its own boundaries and associated instructions. Without this structure, the model might mix operations or misunderstand which instruction applies to which content.

On the flip side, don't add delimiters just because you can 🚫. For simple, single-task prompts where you're asking one straightforward question about one piece of text, delimiters add unnecessary complexity. They're a tool for when structure matters, not a universal best practice. The key is recognizing when the boundaries between different parts of your input actually matter for getting the right output.

---

## 🪜 1.4 Tactic: List the Specific Steps Necessary to Complete a Task

Some tasks are most effectively outlined as a series of steps. Clearly detailing each step helps the model follow instructions more easily, reduces ambiguity, ensures proper sequence, and improves accuracy. Let's see this in action! 📋

### ❌ Example: Without Step-by-Step Instructions

In [12]:
# Without clear steps - general instruction
print("📝 WITHOUT STEP-BY-STEP INSTRUCTIONS:")
print("=" * 70)

text_to_process = """Artificial intelligence (AI) has the potential to revolutionize various industries by automating tasks, providing insights from big data, and improving decision-making processes. AI technologies, such as machine learning, natural language processing, and computer vision, can enhance productivity, efficiency, and innovation. However, the widespread adoption of AI also raises ethical and societal concerns, including issues of privacy, security, and the potential displacement of jobs. To fully harness the benefits of AI, it is crucial to address these challenges through thoughtful regulation, interdisciplinary research, and collaboration between stakeholders from different sectors. Additionally, continuous learning and adaptation are essential to keep up with the rapid advancements in AI and to ensure its responsible and beneficial integration into society. As AI continues to evolve, it will play an increasingly significant role in shaping the future of work, healthcare, transportation, and many other fields."""

prompt_no_steps = f"""Summarize this text in one sentence and provide key points.

{text_to_process}"""

response_no_steps = client.responses.create(
    model=MODEL,
    input=prompt_no_steps
)

print(response_no_steps.output_text)
print("\n" + "=" * 70 + "\n")

📝 WITHOUT STEP-BY-STEP INSTRUCTIONS:


One-sentence summary:
AI has transformative potential to automate tasks, extract insights, and enhance decision-making across industries, but realizing its benefits requires addressing ethical and societal concerns through regulation, interdisciplinary research, collaboration, and ongoing adaptation to ensure responsible integration.

Key points:
- AI technologies (machine learning, natural language processing, computer vision) can boost productivity, efficiency, and innovation by automating tasks and unlocking big-data insights.
- Benefits span multiple sectors, including work, healthcare, and transportation.
- Major concerns include privacy, security, and potential job displacement.
- Addressing these concerns needs thoughtful regulation, interdisciplinary research, and collaboration among stakeholders from different sectors.
- Continuous learning and adaptation are essential to keep pace with rapid AI advancements and to ensure responsible use.




### ✅ Example: With Step-by-Step Instructions

In [13]:
# With clear steps - structured instructions
print("✅ WITH STEP-BY-STEP INSTRUCTIONS:")
print("=" * 70)

prompt_with_steps = f"""Follow these step-by-step instructions to respond to the text below:

Step 1 - Provide a one-sentence summary of this text, starting with 'Summary: '.
Step 2 - Extract the key points from the text in bullet points, starting with 'Key Points: '.

Text:
{text_to_process}"""

response_with_steps = client.responses.create(
    model=MODEL,
    input=prompt_with_steps
)

print(response_with_steps.output_text)

✅ WITH STEP-BY-STEP INSTRUCTIONS:


Summary: AI has the potential to transform industries by automating tasks, deriving insights from big data, and improving decision-making, while its widespread adoption also raises ethical and societal concerns that must be addressed through regulation, collaboration, and continuous learning to ensure responsible and beneficial integration.

Key Points:
- AI can revolutionize industries by automating tasks, providing insights from big data, and improving decision-making.
- Technologies like machine learning, natural language processing, and computer vision boost productivity, efficiency, and innovation.
- Widespread adoption raises ethical and societal concerns, including privacy, security, and potential job displacement.
- Addressing these challenges requires thoughtful regulation, interdisciplinary research, and collaboration among stakeholders from different sectors.
- Continuous learning and adaptation are essential to keep pace with rapid AI advancements and ensure responsible int

### 🎯 When Step-by-Step Instructions Help

Step-by-step instructions become valuable when you're dealing with **multi-part tasks** where sequence and structure matter 🔄. If you're building a data analysis pipeline where you need to first extract information, then categorize it, then format it in a specific way, breaking this into numbered steps ensures the model follows your intended workflow rather than improvising its own approach. This is especially important when each step depends on the completion of previous steps, or when you want specific formatting and labels for each part of the output.

Complex procedural tasks benefit enormously from this approach 🏗️. Imagine you're asking the model to review a document: check for errors, suggest improvements, rewrite problematic sections, and provide a final assessment. Without explicit steps, the model might mix these activities or skip some entirely. With clear steps, you get organized, predictable output where each task is addressed in order. This makes the results easier to parse programmatically and helps ensure nothing is missed.

However, don't overuse this technique for simple questions ⚠️. If you're asking "What's the capital of France?" you don't need to write "Step 1: Identify the question. Step 2: Recall the capital. Step 3: State the answer." That's overkill! Save step-by-step instructions for when the task genuinely involves multiple distinct operations. 

💡 **Rule of Thumb**: If you can explain your task in **three or more distinct steps**, this tactic will likely improve results. For single-step tasks, keep it simple!

---

## 📏 1.5 Tactic: Define the Desired Output Length

You can request the model to generate outputs of a specific target length, such as a certain number of **words, sentences, paragraphs, or bullet points**. This ensures responses are tailored to your exact needs and provides better control over detail and readability. ✂️

⚠️ **Note**: Asking the model to produce an exact word count may not be highly precise. It is more reliable in generating outputs with a specified number of **paragraphs or bullet points**.

### ❌ Example: Without Length Specification

In [14]:
# Without length specification
print("📝 WITHOUT LENGTH SPECIFICATION:")
print("=" * 70)

text_to_summarize = """Artificial intelligence (AI) is transforming various industries by automating tasks, providing insights through data analysis, and enhancing decision-making processes. Machine learning, a subset of AI, allows systems to learn and improve from experience without being explicitly programmed. This technology is used in numerous applications, including healthcare for predictive diagnostics, finance for fraud detection, and transportation for optimizing routes. Despite its benefits, AI also raises ethical concerns, such as data privacy, algorithmic bias, and the potential for job displacement. Addressing these issues requires careful consideration and the implementation of ethical guidelines to ensure AI's responsible use and development."""

response_no_length = client.responses.create(
    model=MODEL,
    input=f"Summarize the following text:\n\n{text_to_summarize}"
)

print(response_no_length.output_text)
print("\n" + "=" * 70 + "\n")

📝 WITHOUT LENGTH SPECIFICATION:


Artificial intelligence automates tasks, analyzes data, and improves decision-making across industries, with machine learning enabling systems to learn from experience. It is applied in healthcare for predictive diagnostics, finance for fraud detection, and transportation for route optimization. However, AI raises ethical concerns such as privacy, algorithmic bias, and potential job displacement, necessitating thoughtful guidelines and responsible development.




### ✅ Example: With Length Specification

In [15]:
# With length specification - 2 bullet points
print("✅ WITH LENGTH SPECIFICATION (2 bullet points):")
print("=" * 70)

response_with_length = client.responses.create(
    model=MODEL,
    input=f"Summarize the following text in exactly 2 short bullet points:\n\n{text_to_summarize}"
)

print(response_with_length.output_text)

✅ WITH LENGTH SPECIFICATION (2 bullet points):


- AI automates tasks, analyzes data, and supports decision-making; ML enables learning from experience and powers applications in healthcare, finance, and transportation.
- It raises ethical concerns such as privacy, bias, and job displacement, requiring guidelines for responsible AI development and use.


### 🎯 When to Specify Length

Specifying output length becomes essential when **space constraints are real and meaningful** 📐. If you're generating social media posts with strict character limits 🐦, writing executive summaries that must fit on one page 📄, or creating bullet points for a presentation slide 📊, telling the model exactly how much space it has prevents the frustration of getting back text that's too long or too short for your needs. This is practical control that saves you editing time!

Length specifications work best when you use **structural units** like sentences, paragraphs, or bullet points rather than exact word counts. The model is quite good at producing "3 bullet points" or "2 short paragraphs" but less precise with "exactly 150 words." It will get close, but don't expect exact word counts—there's some natural variation in how the model interprets and generates text.

The real value of length control is **matching output to purpose** 🎯. A detailed technical report might need multiple paragraphs of explanation, while a quick reference guide needs just a few sentences. An internal memo might be fine at any reasonable length, but a tweet absolutely must fit Twitter's character limit! Use length specifications when the format genuinely matters for your application, not just as arbitrary constraints.

---

## 🚫 1.6 Tactic: Negative Prompting

Negative prompting is the technique of explicitly telling the model what **NOT** to do. While most prompt engineering focuses on what you want the model to do, negative prompting adds constraints by specifying unwanted behaviors, formats, or content types. This technique complements positive instructions by preventing common pitfalls you've observed in practice. 🛡️

Think of negative prompting as **preventive medicine** 💊 for your prompts. When you've worked with a model and noticed it consistently makes certain mistakes—like using jargon when you want simple language, or providing overly technical explanations when your audience is non-technical—negative prompting lets you explicitly block these tendencies. It's reactive in the sense that it addresses observed problems, but preventive because it stops those problems before they happen in future outputs.

The power of negative prompting comes from its **specificity** 🎯. Rather than hoping the model will intuitively understand your audience's level or preferences, you state boundaries clearly. This is particularly valuable when the default behavior of the model doesn't quite match your needs, or when you're working in a domain where certain types of content or assumptions would be problematic.

### ❌ Example: Without Negative Prompting

In [16]:
# Without negative prompting - model might use jargon
print("📝 WITHOUT NEGATIVE PROMPTING:")
print("=" * 70)

response_no_negative = client.responses.create(
    model=MODEL,
    input="Explain how machine learning works."
)

print(response_no_negative.output_text)
print("\n" + "=" * 70 + "\n")

📝 WITHOUT NEGATIVE PROMPTING:


Here’s a concise, practical way to think about how machine learning (ML) works.

- The basic idea
  - ML tries to learn a function that maps inputs (features) to outputs (labels or predictions) from examples. Instead of writing explicit rules, you let a computer figure the rules from data.

- Core ingredients
  - Data: examples consisting of inputs and (for supervised learning) correct outputs.
  - Model: a parameterized function fθ(x) that makes predictions. The parameters θ are what you learn.
  - Objective (loss) function: a math measure of how wrong the model is on the training data (e.g., mean squared error for regression, cross-entropy for classification).
  - Optimization algorithm: a method to adjust θ to reduce the loss (e.g., gradient descent).

- A typical workflow (supervised learning)
  1) Define the goal: what should the model predict?
  2) Collect data and split it into training and test (and sometimes validation) sets.
  3) Preprocess and possibly engineer features (nor

### ✅ Example: With Negative Prompting

In [17]:
# With negative prompting - explicitly avoid unwanted elements
print("✅ WITH NEGATIVE PROMPTING:")
print("=" * 70)

prompt_with_negative = """Explain how machine learning works.

DO NOT use technical jargon.
DO NOT assume prior programming knowledge.
DO NOT include mathematical formulas.
DO NOT use acronyms without explaining them first."""

response_with_negative = client.responses.create(
    model=MODEL,
    input=prompt_with_negative
)

print(response_with_negative.output_text)

✅ WITH NEGATIVE PROMPTING:


Machine learning is a way for computers to get better at a task by studying examples, instead of being told every exact rule they must follow.

Think about teaching a child to recognize cats in photos. You show lots of pictures labeled “cat” and “not cat.” Over time, the child starts to notice patterns (often furry faces, whiskers, pointy ears, etc.) and learns to tell cats from other things. Machine learning works in the same idea, but with a computer.

How it works in plain terms:

- Step 1: Gather examples
  - You collect examples that show what you want the computer to do. For instance, a batch of photos labeled as “cat” or “not cat,” or a set of emails labeled as “spam” or “not spam.”

- Step 2: Give the computer a simple way to use those examples
  - You present the computer with a straightforward method to look at the examples and form a rule or pattern. Think of it as a very simple judgment habit the computer can adjust.

- Step 3: Practice and adjust
  - The computer tries to 

### 🎯 When to Use Negative Prompting

Negative prompting excels when you're trying to reach specific audiences who need information presented in particular ways 👥. If you're creating educational content for beginners 🎓, you might need to prevent the model from using technical terminology or assuming background knowledge they don't have. If you're writing for a general audience 🌍, you might want to avoid academic language or industry-specific jargon that would alienate readers. These negative constraints shape the communication style to match your audience's needs.

**Format control** is another strong use case 📐. Sometimes you need narrative prose, not bulleted lists. Other times you want direct answers without lengthy preambles. Negative prompting lets you specify these format preferences explicitly, especially useful when the model's default formatting doesn't match your application's requirements. If you're generating content for a specific platform or document type, negative instructions help maintain consistency.

The technique becomes especially powerful when you've **iterated on prompts** and noticed specific patterns of unwanted behavior 🔄. Maybe the model keeps going off on tangents, or includes unnecessary caveats, or makes assumptions about context that aren't valid for your use case. Negative prompting is your way of course-correcting based on observed patterns. It's less about theoretical best practices and more about practical adjustments based on real usage.

⚠️ Don't use negative prompting as your first approach for every prompt. Start with clear positive instructions about what you want. Add negative constraints when you've identified specific problems that need prevention. Overusing negative prompting can make prompts verbose and may even draw the model's attention to things you're trying to avoid. It's most effective when targeted at genuine, observed issues rather than theoretical concerns.

💡 **Key Insight**: Negative prompting is especially powerful when you've iterated on prompts and noticed specific patterns of unwanted behavior. It's reactive and preventive—use it to address **real problems** you've observed, not hypothetical ones!

---

# 2️⃣ Providing Reference Text

One effective way to improve the reliability and accuracy of AI responses is to provide **reference text** 📚 that the model should use when generating answers. This grounds the model's responses in verified information rather than relying solely on its training data.

## 📖 2.1 Tactic: Guide the Model to Answer Using Reference Text

We can instruct the model to answer questions based on a provided reference text. By supplying the model with trusted and relevant information, you can direct it to use this information to generate accurate answers, reducing the likelihood of **hallucinations** 👻 or misinformation.

### ❌ Example: Without Reference Text

In [18]:
# Without reference text - model relies on training data
print("📝 WITHOUT REFERENCE TEXT:")
print("=" * 70)

response_no_ref = client.responses.create(
    model=MODEL,
    input="How many deaths worldwide are attributed to household air pollution?"
)

print(response_no_ref.output_text)
print("\n" + "=" * 70 + "\n")

📝 WITHOUT REFERENCE TEXT:


About 3.8 million people die each year worldwide from household air pollution (from burning solid fuels like wood, charcoal, coal, dung for cooking/heating). Note that estimates vary by year and data source, but this is the commonly cited figure from WHO. If you want, I can look up the latest numbers or provide other estimates.




### ✅ Example: With Reference Text

In [19]:
# With reference text - model uses provided information
print("✅ WITH REFERENCE TEXT:")
print("=" * 70)

prompt_with_ref = """Refer to the provided text enclosed within XML tags to answer the questions. If the answer is not found in the text, respond with 'I could not find an answer.'

<article>
The keyword co-occurrence network reveals a strong correlation between air pollution and the burden of disease. As air pollution intensifies, there is a commensurate rise in the global burden of disease. In 2019, the toll of fatalities attributed to air pollution across the globe amounted to 6.67 million. Indoor air pollution exhibits a strong correlation with family health. The associated effects and disease burden garner escalating attention from academics and governments. Based on the statistical data derived from the Global Burden of Disease, household air pollution from solid fuels has caused 2.3 million deaths worldwide. Among them, 1.06 million deaths are attributed to cardiovascular diseases, 420,000 deaths are caused by respiratory infections and tuberculosis, while 398,000 deaths are associated with chronic respiratory diseases. According to the World Health Organization, the utilization of solid fuels and incomplete combustion leading to household air pollution will result in 3.2 million premature deaths worldwide. Thus, when considering the impact of air pollution on health, indoor space is a crucial microenvironment that needs to be given ample attention.
</article>

Question: How many deaths worldwide are attributed to household air pollution?"""

response_with_ref = client.responses.create(
    model=MODEL,
    input=prompt_with_ref
)

print(response_with_ref.output_text)

✅ WITH REFERENCE TEXT:


2.3 million deaths.


### 🎯 When to Use Reference Text

Reference text becomes essential when you have **authoritative sources** 📑 and need responses grounded in specific information rather than general knowledge. This is the technique to reach for when accuracy and verifiability matter more than creative interpretation. If you're building a Q&A system over company documentation, policy manuals, research papers, or any corpus where answers must reflect the exact content of your materials, reference text constrains the model to work within those boundaries.

The primary benefit is **reducing hallucinations** 🚫👻—those moments when models confidently state plausible-sounding information that isn't actually true. By providing reference text and instructing the model to answer only from that text, you dramatically reduce this risk. The model can still fail to find relevant information or misinterpret the text, but it's much less likely to invent facts entirely. This makes reference text valuable for applications where trust and accuracy are paramount: customer support systems 🎧, compliance tools ⚖️, research assistants 🔬.

**Company-specific information** 🏢 is another ideal use case. The model's training data doesn't include your internal processes, product specifications, or organizational policies. Providing this information as reference text lets you leverage the model's language understanding while ensuring responses reflect your specific context. This bridges the gap between the model's general capabilities and your particular needs.

However, don't use reference text for creative tasks 🎨 or general knowledge questions where you actually want the model to draw on its broad training. If you're asking for brainstorming ideas, creative writing, or synthesis of widely-known information, providing reference text might unnecessarily constrain the model. The technique is about **focus and accuracy**, not about enhancing creativity or breadth.

---

## 📎 2.2 Tactic: Guide the Model to Answer with Citations from Reference Texts

You can instruct the model to include **citations** in its answers by referencing specific passages from provided documents. This allows citations in the output to be programmatically verified through string matching, significantly improving the trustworthiness of the model's outputs. ✅

In [20]:
# With citations from reference text
print("✅ WITH CITATIONS:")
print("=" * 70)

prompt_with_citations = """You will be given a document delimited by XML tags and a question. Your task is to answer the question using only the information from the provided document and to cite the relevant passages. 

If the document does not contain enough information to answer the question, write: 'I could not find an answer.' 

Any answer provided must include a citation. Use the following format to cite relevant passages: ({"citation": ...})

<article>
The keyword co-occurrence network reveals a strong correlation between air pollution and the burden of disease. As air pollution intensifies, there is a commensurate rise in the global burden of disease. In 2019, the toll of fatalities attributed to air pollution across the globe amounted to 6.67 million. Indoor air pollution exhibits a strong correlation with family health. The associated effects and disease burden garner escalating attention from academics and governments. Based on the statistical data derived from the Global Burden of Disease, household air pollution from solid fuels has caused 2.3 million deaths worldwide. Among them, 1.06 million deaths are attributed to cardiovascular diseases, 420,000 deaths are caused by respiratory infections and tuberculosis, while 398,000 deaths are associated with chronic respiratory diseases. According to the World Health Organization, the utilization of solid fuels and incomplete combustion leading to household air pollution will result in 3.2 million premature deaths worldwide. Thus, when considering the impact of air pollution on health, indoor space is a crucial microenvironment that needs to be given ample attention.
</article>

Question: How many deaths worldwide are attributed to household air pollution?"""

response_with_citations = client.responses.create(
    model=MODEL,
    input=prompt_with_citations
)

print(response_with_citations.output_text)

✅ WITH CITATIONS:


2.3 million deaths worldwide. ({"citation": "Based on the statistical data derived from the Global Burden of Disease, household air pollution from solid fuels has caused 2.3 million deaths worldwide."})


### 🔍 Why Citations Matter

Citations transform AI-generated responses from unverifiable claims into **traceable information** 📍. In professional contexts where accountability matters—legal analysis ⚖️, medical information 🏥, financial advice 💰, academic research 🎓—being able to trace a statement back to its source is often required, not optional. Citations allow readers to verify information, check context, and assess the reliability of sources, which is fundamental to trust.

The practical benefit extends beyond trust to **functionality** ⚙️. When the model cites specific passages, you can programmatically verify those citations by checking if the quoted text actually appears in the source documents. This lets you build quality checks into your application—flagging responses where citations don't match sources, or filtering out answers that lack proper citations. This kind of automated verification is impossible without explicit citations!

Academic and research applications demand citations by nature, but the need extends to business contexts too 💼. When your company's AI assistant answers questions about policies or procedures, having citations that point to specific policy sections protects both the company and users. If there's ever a dispute about what was communicated, citations provide an **audit trail** 📋. In regulated industries like healthcare, finance, or legal services, this traceability can be a compliance requirement, not just a nice-to-have feature.

---

# 3️⃣ Few-Shot Prompting

Few-shot prompting is a powerful technique that leverages **examples** 📝 to guide the model's behavior. Rather than just telling the model what to do, you **show** it examples of what you want! ✨

## 🧠 Understanding Few-Shot Prompting

### 💭 What is Few-Shot Learning?

Few-shot prompting works by providing the AI model with a small number of examples that demonstrate the task you want it to perform. These examples—often called "**shots**" 🎯—serve as templates that show the model the pattern you're looking for. It's fundamentally different from **zero-shot prompting** (where you just describe the task without examples) and from **fine-tuning** (where you permanently train the model on many examples to specialize its behavior).

The psychology behind few-shot learning is intuitive 🧒. Imagine teaching a child to identify different types of clouds. You could describe the characteristics of cumulus clouds in abstract terms, or you could simply point at the sky and say "see those fluffy, cotton-like clouds? Those are cumulus." Show them a few more examples, and they'll start recognizing cumulus clouds on their own. Few-shot prompting works similarly—the model sees patterns in your examples and generalizes those patterns to new cases.

When the model sees examples, it picks up on subtle aspects that might be hard to describe explicitly: the **tone** 🎵, the **level of detail** 🔍, the **structure** 🏗️ of the response, the way certain types of inputs map to outputs. Examples communicate these nuances more effectively than lengthy descriptions. A single well-chosen example can convey what would take paragraphs to explain, and multiple examples reinforce the pattern while showing acceptable variations.

### 🔢 How Many Examples Should You Provide?

The right number of examples depends on your task's complexity and how consistently the model needs to perform. For **simple tasks** with clear patterns—like basic classification or format conversion—**2-3 examples** are often sufficient to establish the pattern. The model picks up on straightforward patterns quickly, and additional examples provide diminishing returns.

More **nuanced tasks** requiring subtle distinctions benefit from **5-10 examples**. If you're doing sentiment analysis that distinguishes between frustrated, angry, and disappointed, or if you're extracting structured data where edge cases matter, more examples help the model understand the boundaries between categories and handle ambiguous cases better. Each additional example can clarify a different aspect of the task or demonstrate how to handle a particular edge case.

However, more examples aren't always better ⚠️. Beyond a certain point, you hit **diminishing returns**—the model isn't learning much from additional examples, but you're paying for those extra tokens in every request 💸. More examples also make prompts more complex and harder to maintain. If results are good with three examples, don't add seven more just because you can. The sweet spot is the **minimum number of examples** that consistently produces the quality you need.

💡 **Pro Tip**: If results are good with 3 examples, don't add more just because you can!

### ⭐ Choosing Good Examples

The **quality** of your examples matters more than quantity! Good examples should cover the range of cases you expect to encounter—not just typical cases, but also **edge cases** and potentially tricky situations. If you're classifying customer feedback, don't just show positive and negative examples; include ambiguous ones, mixed sentiment, and edge cases like sarcasm or context-dependent meaning. This helps the model handle real-world messiness.

**Diversity within consistency** is key 🔑. Your examples should show variety in inputs while maintaining consistent formatting and style in outputs. If all your examples are too similar, the model might latch onto superficial patterns rather than the underlying logic. But if examples are inconsistent in format or approach, the model gets confused about what pattern to follow. Strike a balance between showing the range of inputs you'll encounter and maintaining a clear, consistent pattern in how outputs should look.

What makes a good example? ✨
- **Clear and unambiguous** - even a human should immediately understand the task
- **Representative** of real cases you'll encounter, not artificial or oversimplified
- **Demonstrates** one or more important aspects of the task—format, tone, level of detail, how to handle a specific type of input

### 🎯 When to Use Few-Shot Prompting

Few-shot prompting excels at **pattern recognition tasks** where showing is easier than telling! Classification problems 🏷️—sentiment analysis, topic categorization, intent detection—are natural fits. Extraction tasks where you need specific information pulled out in a particular format benefit from examples that show exactly what you want and how it should be structured.

When **zero-shot prompting gives inconsistent results** 🎲—sometimes the format is right, sometimes wrong, or the tone varies unpredictably—adding examples usually stabilizes performance. Tasks with **specific formatting requirements** are perfect for few-shot learning. If you need JSON output with particular field names, or CSV data with specific delimiters, or markdown with a particular structure, showing examples of correctly formatted output is far more effective than trying to describe the format in words.

However, don't use few-shot prompting for simple factual questions where zero-shot works fine ❌. If "What's the capital of France?" gets you "Paris," you don't need examples. Creative writing tasks also generally don't benefit from examples unless you're trying to match a very specific style. For open-ended creative work, examples might constrain the model in unhelpful ways, making outputs feel derivative rather than original.

### 💰 Cost and Performance Trade-offs

Few-shot prompting has a real cost: every example adds tokens to your prompt, and you pay for those tokens in **every single request** 💵. If your base prompt is 50 tokens and you add five examples that total 200 tokens, you've **quintupled** the input cost of each request. Over thousands or millions of requests, this adds up significantly.

The decision becomes a balance between **accuracy improvement** and **cost increase** ⚖️. If adding examples boosts your accuracy from 70% to 95%, that might be worth the extra cost—fewer errors means less manual review, fewer user complaints, better user experience. But if examples only improve accuracy from 95% to 97%, you need to consider whether that marginal gain justifies the added expense. Sometimes better prompt engineering—clearer instructions, better structure, strategic use of other techniques—can achieve similar results without the per-request cost of examples.

---

## 🧪 Practical Example: Sentiment Analysis

Let's compare zero-shot and few-shot prompting using a challenging domain: analyzing **Gen-Z slang** and informal language! 😎

### ❌ Zero-Shot Prompting

In [21]:
# Zero-shot: Just ask without examples
print("📝 ZERO-SHOT PROMPTING:")
print("=" * 70)

zero_shot_prompt = "Classify the following Gen-Z style statement as Positive, Negative, or Neutral: 'This party is mid fr fr no cap.'"

response_zero_shot = client.responses.create(
    model=MODEL,
    input=zero_shot_prompt
)

print(response_zero_shot.output_text)
print("\n" + "=" * 70 + "\n")

📝 ZERO-SHOT PROMPTING:


Negative




### ✅ Few-Shot Prompting

In [22]:
# Few-shot: Provide examples first
print("✅ FEW-SHOT PROMPTING:")
print("=" * 70)

few_shot_prompt = """You are an AI trained to classify Gen-Z style statements as Positive, Negative, or Neutral.

Here are some examples:

1. Statement: 'That fit is fire, you slayed'
   Sentiment: Positive

2. Statement: 'I'm lowkey stressed about this exam'
   Sentiment: Negative

3. Statement: 'This new album straight bussin, no cap'
   Sentiment: Positive

Now, classify the following statement:
Statement: 'It's giving boring vibes ngl'
Sentiment:"""

response_few_shot = client.responses.create(
    model=MODEL,
    input=few_shot_prompt
)

print(response_few_shot.output_text)

✅ FEW-SHOT PROMPTING:


Negative


Notice how the few-shot version performs better with informal language! 🎉 The examples showed the model how to interpret Gen-Z slang like "mid" (mediocre), "fr fr" (for real), "no cap" (no lie), and "bussin" (excellent). Without examples, the model might struggle with these terms or misinterpret the sentiment. The examples don't just show correct answers—they demonstrate the **pattern** of how informal language maps to sentiment categories.

This improvement comes from the examples clarifying the specific challenge: **understanding slang and informal expressions in context** 💬. The few-shot version effectively taught the model this specialized task through demonstration rather than description, which is exactly when few-shot prompting shines! ✨

---

# 4️⃣ Chain-of-Thought Prompting

Chain-of-thought prompting asks the model to **show its reasoning process** step-by-step 🧮, making the logic explicit rather than jumping directly to a conclusion. This technique significantly improves performance on complex reasoning tasks!

## 🧩 Understanding Chain-of-Thought Prompting

### 💡 What is Chain-of-Thought Prompting?

Chain-of-thought (CoT) prompting is the technique of asking the model to **show its work** 📝—to explicitly walk through its reasoning process before arriving at an answer. Rather than asking "What is the answer?" you ask "What is the answer and **how did you arrive at it**?" This seemingly simple change has profound effects on the model's performance, especially for problems requiring multiple steps of reasoning.

The technique emerged from research showing that language models perform dramatically better on complex problems when they generate **intermediate reasoning steps** 🪜. It's like showing your work in math class—the act of writing out each step catches errors, clarifies logic, and makes it easier to verify the answer. When models jump directly to conclusions, they're more likely to make mistakes in multi-step problems. When they reason step-by-step, they essentially self-correct and build more reliable solutions.

Think about how you'd solve a complex problem yourself 🤔. You don't usually arrive at the answer in one intuitive leap. You break it down: first figure out this component, then use that result to calculate the next part, then combine everything for the final answer. Chain-of-thought prompting asks the model to follow a similar process, making its reasoning **transparent and verifiable** ✅.

### 🎯 When to Use Chain-of-Thought

Chain-of-thought prompting is your go-to technique for **multi-step reasoning problems** 🧮. Math problems, logic puzzles, analytical tasks requiring multiple inferential steps—these are prime candidates. If you're asking the model to analyze data and draw conclusions, or to solve problems where the path from question to answer involves several logical steps, CoT dramatically improves reliability.

The technique becomes especially valuable when you need to **verify the reasoning**, not just accept the answer 🔍. In educational contexts 🎓, showing the reasoning process helps students learn. In professional contexts like data analysis or decision support 💼, stakeholders often need to understand **how** conclusions were reached, not just **what** they are. Chain-of-thought makes the model's logic transparent and auditable.

**Debugging** why an answer might be wrong 🐛 becomes much easier with chain-of-thought. If the model gets a problem wrong, seeing its reasoning process often immediately reveals where it went astray. Maybe it misinterpreted a key piece of information, or made a calculation error in step 3, or applied the wrong rule. Without seeing the steps, you're left guessing why the answer is wrong.

### 🚫 When to Skip Chain-of-Thought

Not every task benefits from explicit reasoning. **Simple factual lookups** don't need it—"What's the capital of Japan?" doesn't require showing work; the answer is "Tokyo." **Creative writing tasks** 🎨 often suffer from step-by-step reasoning because it breaks the flow and makes output feel mechanical rather than natural. If you want the model to write a story or generate marketing copy, asking it to show its reasoning about word choices would be counterproductive.

When you only care about the **final answer** and the reasoning would just add unnecessary verbosity, skip CoT. Some applications need quick, concise responses ⚡, and multi-paragraph reasoning explanations would frustrate users. If your users just want to know "yes" or "no," they don't need to see three paragraphs explaining how the model arrived at its answer.

### 💰 Cost Implications

The cost impact of chain-of-thought prompting is substantial and worth understanding before committing to it. A direct answer might be 50 tokens, while a chain-of-thought response explaining the same answer could be **200-300 tokens**. That's **4-6 times** the output cost per request! For a high-volume application, this difference compounds quickly.

However, this cost-benefit calculation isn't just about token counts ⚖️. Chain-of-thought often produces **more accurate answers**, especially for complex problems. If direct answers are wrong 20% of the time, requiring manual review or corrections, while CoT answers are wrong only 5% of the time, the extra token cost might be offset by reduced error handling. You're trading increased per-request costs for better accuracy, which could mean lower total operational costs when you factor in error correction and customer support! 📊

---

## 🧮 Practical Example: Multi-Step Math Problem

In [23]:
# Chain-of-thought prompt with example showing reasoning steps
print("✅ CHAIN-OF-THOUGHT PROMPTING:")
print("=" * 70)

cot_prompt = """Question: A train travels 90 kilometers in 2 hours. How long will it take to travel 225 kilometers at the same speed?

Answer:
Let's break this down step by step:

1. First, we need to calculate the speed of the train. To do that, we divide the distance traveled by the time taken.
   So, the speed of the train is 90 kilometers divided by 2 hours, which equals 45 kilometers per hour.

2. Now, we know the speed of the train is 45 kilometers per hour. Next, we calculate how long it will take to travel 225 kilometers.

3. To do that, we divide 225 kilometers by the speed of 45 kilometers per hour.

4. 225 kilometers divided by 45 kilometers per hour equals 5 hours.

So the answer is 5 hours.

---

Question: A car travels 150 kilometers using 12 liters of fuel. If the car needs to travel 450 kilometers, how much fuel will it require? Assume the fuel efficiency remains constant.

Answer:"""

response_cot = client.responses.create(
    model=MODEL,
    input=cot_prompt
)

print(response_cot.output_text)

✅ CHAIN-OF-THOUGHT PROMPTING:


36 liters.

Reason: fuel consumption is proportional to distance. 12 L for 150 km means 12/150 = 0.08 L per km. For 450 km: 0.08 × 450 = 36 L. (Or x = 12 × (450/150) = 36 L.)


The chain-of-thought approach transformed how the model tackled this problem! 🎯 By showing an example that breaks down the problem into clear steps—calculate the rate, apply the rate to the new distance, state the answer—we taught the model to follow the same methodical process. The resulting answer isn't just correct; it's **verifiable** ✅. Anyone can follow the logic and check each step.

This is when chain-of-thought truly shines: when the **path to the answer is as important as the answer itself** 🛤️. The explicit reasoning catches potential errors (if step 2 doesn't follow from step 1, you'll see that immediately), makes the solution easier to trust, and helps users understand the logic rather than just accepting a number. For educational applications 🎓, troubleshooting systems 🔧, analytical tools 📊, or any context where decisions need to be justified, chain-of-thought prompting is worth the extra tokens!

---

# 5️⃣ Self-Ask Prompting

Self-ask prompting is an **advanced technique** 🚀 where the model breaks down complex questions into sub-questions, answers each one, and then synthesizes the results. It's like watching the model conduct a structured research process! 🔬

## 🧠 Understanding Self-Ask Prompting

### 💭 What is Self-Ask Prompting?

Self-ask prompting teaches the model to **decompose complex questions** into simpler sub-questions before attempting to answer them. Rather than tackling a multi-faceted question directly, the model asks itself follow-up questions that address different aspects of the original question, answers each one, and then synthesizes these partial answers into a comprehensive response. It's an **internal dialogue** 💬 or structured brainstorming session made explicit.

The technique differs fundamentally from chain-of-thought prompting, though they're often confused 🤔. Chain-of-thought shows the **steps** to reach an answer—it's a linear path through a problem. Self-ask, on the other hand, breaks the **question itself** into multiple sub-questions first. It's about problem decomposition before problem-solving. If chain-of-thought is "here's how to get from A to B," self-ask is "to understand B, we need to first understand X, Y, and Z, so let's explore each of those."

Imagine you're researching a complex topic like "How will climate change affect global food security?" 🌍 You wouldn't just sit down and write an answer. You'd break it down: How will climate change affect crop yields? How will it affect water availability? What about extreme weather events? How might adaptation strategies mitigate impacts? Self-ask prompting mirrors this natural research process, making the model's approach to complex questions more **structured and thorough** ✨.

### 🎯 When to Use Self-Ask

Self-ask excels with **genuinely complex, multi-faceted questions** 🌐 that require analyzing different dimensions before reaching a conclusion. When a question is so broad that a direct answer would inevitably miss important aspects, self-ask ensures comprehensive coverage by forcing systematic decomposition. Questions like "What are the implications of autonomous vehicles for urban planning?" 🚗 or "How might artificial intelligence affect income inequality?" 💰 have so many angles that breaking them down into sub-questions leads to more thorough analysis.

The technique shines when the initial question is **too broad or vague** 🌫️ to answer well directly. Self-ask essentially forces clarification through decomposition. By breaking "What are the effects of social media?" into specific sub-questions about mental health, political discourse, business models, and social connections, the model transforms a nebulous query into concrete answerable components.

**Research-style questions** 📚 with multiple dimensions benefit enormously from self-ask. If you're building a system that explores topics in depth, or generating research reports, or analyzing complex situations, self-ask provides the structured approach that ensures nothing important gets overlooked. The model essentially follows a research protocol: identify key sub-questions, address each one, synthesize findings.

### 🔄 Self-Ask vs Chain-of-Thought

Understanding when to use self-ask versus chain-of-thought requires grasping their fundamental difference 🎓:

**Chain-of-Thought** 🪜: For problems with a relatively clear solution path that requires multiple steps. You know what you're solving for; you just need to show the work. It's linear and procedural—step 1, step 2, step 3, answer.

**Self-Ask** 🔍: For questions where the challenge is in the question's complexity itself, not just in the solution steps. You need to break down what's being asked before you can answer it. It's more exploratory and multidimensional. The model identifies aspects of the question, addresses each aspect, and then synthesizes. It's not a linear path but rather a structured exploration.

💡 **How to Choose**: If someone asks "What's 15% of 240?" use chain-of-thought to show the calculation steps. If someone asks "How might universal basic income affect employment, government budgets, and social welfare?" use self-ask to systematically address each dimension.

### ⚠️ Limitations and When Not to Use

Self-ask prompting is **token-expensive** 💸—arguably the most expensive technique in common use. You're generating multiple questions and multiple answers for each original question. A direct response might be 100 tokens, while a self-ask response could easily be **500-1000 tokens**. The model's asking and answering its own follow-up questions, each adding to the output length. For high-volume applications, this cost adds up quickly!

Don't use self-ask for **simple or focused questions** 🚫. If someone asks "What's the capital of Brazil?" breaking that down into sub-questions is absurd overkill. Save self-ask for questions where the complexity genuinely demands decomposition. Using it for routine queries just wastes tokens and produces verbose outputs that obscure simple answers.

The technique can produce outputs so comprehensive that they become **overwhelming** 📚. Sometimes users want a concise answer, not an exhaustive exploration. A simple question might get buried under layers of sub-questions and sub-answers. Before choosing self-ask, consider whether your users actually need that level of detail or if it would be information overload.

✅ **Best used judiciously** for genuinely complex analytical questions where thoroughness matters more than brevity. Research reports 📊, strategic analysis 🎯, comprehensive assessments—these are appropriate contexts. Quick information lookups, routine support questions, or any scenario where speed and conciseness matter more than exhaustive coverage should use simpler techniques.

### 🔍 Transparency Benefit

Despite its costs, self-ask has a unique **transparency advantage** 💎. The technique explicitly shows the model's reasoning structure—not just how it arrived at an answer, but **what aspects of the question** it considered important. If the model breaks a question about climate policy into sub-questions about economic impact, environmental effectiveness, political feasibility, and social equity, you immediately see its analytical framework.

This transparency helps identify if the model is **missing crucial aspects** ⚠️. If you ask about implementing a new technology and the model's sub-questions don't include security or privacy considerations, that omission becomes obvious. You can then refine your prompt or follow up specifically on overlooked dimensions. The explicit structure makes gaps visible in a way that direct answers don't.

When you need to understand how conclusions were reached—not just verify computational steps but see the entire analytical structure—self-ask provides **unmatched visibility** 👀 into the model's reasoning process.

---

## 🌍 Practical Example: Complex Policy Question

In [24]:
# Self-ask prompt with example showing the decomposition process
print("✅ SELF-ASK PROMPTING:")
print("=" * 70)

self_ask_prompt = """Question: How could the implementation of a universal basic income affect wealth inequality, job markets, and economic growth in a developed country?

Are follow up questions needed here: Yes.

Follow up: How might universal basic income impact wealth inequality?
Intermediate answer: Universal basic income could potentially reduce wealth inequality by providing a financial safety net for all citizens, ensuring a minimum standard of living.

Follow up: What effect could universal basic income have on job markets?
Intermediate answer: It might reduce the pressure to work in low-wage jobs, potentially leading to increased bargaining power for workers and changes in the labor market structure.

Follow up: How could universal basic income influence economic growth?
Intermediate answer: Universal basic income could stimulate consumer spending and potentially boost economic growth, but it may also lead to inflation or reduced productivity if not implemented carefully.

So the final answer is: The implementation of a universal basic income in a developed country could potentially reduce wealth inequality by providing a financial safety net, alter job markets by changing worker dynamics, and affect economic growth through increased consumer spending, though careful implementation would be crucial to manage potential negative effects like inflation.

---

Question: How might the widespread adoption of autonomous vehicles impact urban planning and infrastructure in major cities over the next 20 years?

Are follow up questions needed here:"""

response_self_ask = client.responses.create(
    model=MODEL,
    input=self_ask_prompt
)

print(response_self_ask.output_text)

✅ SELF-ASK PROMPTING:


Yes. To give a robust assessment of how autonomous vehicles (AVs) could reshape urban planning and infrastructure, it helps to clarify scope, scenarios, and policy context with targeted follow-up questions.

Proposed follow-up questions
- Adoption scenario: What share of vehicles are autonomous, and will AVs be private, ride-hailing fleets, or mixed fleets? What is the expected timeline (e.g., 2030, 2035, 2040)?
- Urban context: How does city density, existing public transit quality, and housing demand influence AV deployment and complementary projects?
- Curb and parking: How will curb space be reallocated (loading zones, pick-up/drop-off, bus lanes, bike lanes) and how much parking demand will AVs reduce?
- Infrastructure needs: What investments are required for road sensors, V2X communications, digital signaling, charging/fueling infrastructure, and data networks?
- Traffic and road design: How might AVs affect congestion, speed patterns, lane usage, and street design (e.g., narrowe

Notice how self-ask transformed the approach to this complex question! 🎯 By providing an example that breaks down a multifaceted policy question into specific sub-questions about different impacts, we taught the model to systematically explore each dimension before synthesizing an answer. The model identified relevant aspects (parking needs, traffic flow, public transit, land use), addressed each one, and then brought them together into a coherent conclusion.

This structured exploration produces **more thorough answers** 📊 than direct questioning would. It ensures the model considers multiple angles rather than focusing on just one aspect. For genuinely complex questions where comprehensive analysis matters, self-ask is worth the significant token investment. But use it selectively—this is your **advanced technique** 🚀 for when simpler approaches won't do justice to the question's complexity!

---

# 🎓 Best Practices & Key Takeaways

## 🎯 Choosing the Right Technique

The art of prompt engineering lies in **matching techniques to tasks** 🎨. When your task needs focused, specific information on a particular topic, the foundation is **detailed prompts** that include relevant context and constraints. Don't ask "tell me about machine learning"—ask "explain how supervised learning differs from unsupervised learning, with one example of each, for someone with basic programming knowledge but no AI background." Specificity transforms generic outputs into useful ones! ✨

If the tone and style of communication matter for your application—customer service 🤝, education 🎓, sensitive subjects 💙—**personas** become your tool. A technical expert persona for professional audiences, an empathetic counselor persona for healthcare contexts, a patient tutor persona for educational content—these shape how information is delivered to match your audience's needs and expectations. Just remember that personas affect **style**, not **knowledge**, so combine them with other techniques when accuracy matters.

When you're handling multiple documents or complex inputs in a single request, **delimiters** 📦 prevent confusion about what text should be processed how. They're particularly crucial for user-generated content that might contain confusing text, or for applications that need to treat different sections of input differently. Skip delimiters for simple single-piece-of-text tasks where boundaries aren't an issue.

Tasks involving multiple sequential steps benefit from explicit **step-by-step instructions** 🪜. If you can break your task into three or more distinct operations—extract this, then analyze that, then format it this way—numbering those steps ensures the model follows your intended workflow. Save this for genuinely multi-part tasks; don't overstructure simple requests.

**Negative prompting** 🚫 becomes valuable when you've observed specific unwanted behaviors you want to prevent. If the model keeps using jargon when you need plain language, or making assumptions about audience knowledge, or including elements you don't want, explicitly stating what NOT to do can be more effective than trying to imply constraints through positive instructions alone.

When you have authoritative **reference documents** 📚 and need responses grounded in specific information rather than general knowledge, provide that reference text explicitly. This dramatically reduces hallucinations 👻 and ensures answers reflect your particular context—company policies, research papers, legal documents, whatever source should ground the responses.

For tasks requiring consistent patterns—classification, extraction, formatting—**few-shot prompting** 🎯 lets you show what you want rather than describe it. Two to five good examples often establish patterns more clearly than paragraphs of description. Use this when zero-shot results are inconsistent or when format requirements are easier to demonstrate than explain.

When you need to verify reasoning, not just trust answers, **chain-of-thought** 🧮 makes the model's logic transparent. Multi-step math problems, analytical tasks, troubleshooting—anything where the path to the answer matters as much as the answer itself. The cost is higher token usage, but the benefit is verifiable, trustworthy reasoning.

For genuinely complex, multifaceted questions where thoroughness matters more than brevity, **self-ask prompting** 🔍 ensures systematic exploration of all relevant dimensions. It's your most expensive technique in terms of tokens, so reserve it for questions that truly demand comprehensive analysis rather than quick answers.

## 🔗 Combining Techniques

The real power emerges when you combine techniques strategically! 💪 You might use **personas with negative prompting** to get both the right tone and avoid specific pitfalls—"you're a patient educator" plus "don't use jargon or assume prior knowledge." **Few-shot examples** can incorporate **chain-of-thought reasoning**, showing not just what outputs should look like but how to reason through each case. **Reference text** can be paired with **step-by-step instructions** for complex document analysis tasks.

However, combining techniques also means combining **costs and complexity** 💰. Each additional technique adds tokens, makes prompts harder to maintain, and can create confusion if techniques conflict. The key is **purposeful combination**—add another technique only when it addresses a specific gap in your results. Start with the simplest approach that might work, then add techniques incrementally when you identify specific deficiencies.

A common effective combination is **detailed prompts + step-by-step instructions** for structured tasks, plus **few-shot examples** to establish format. Another is **personas + reference text** for customer-facing applications that need both appropriate tone and accurate information. The combination should feel natural, with each technique addressing a different aspect of quality—one for accuracy, one for tone, one for format.

## 💭 Final Thoughts

Prompt engineering is fundamentally **iterative** 🔄. You rarely get prompts perfect on the first try. Start with a straightforward attempt, see what works and what doesn't, then refine. Each iteration teaches you something about how the model interprets your instructions and where your prompts need more clarity or different structure. The techniques in this notebook are **tools in your toolkit** 🧰—you'll learn through experience which ones work best for your specific use cases.

What works beautifully for one task might fail for another that seems similar. Prompt engineering requires **experimentation and adaptation** 🧪. Don't assume that because a technique worked well once, it's the right approach for every problem. Be prepared to try different techniques, combine them in new ways, and adjust based on results. The goal is **reliable, useful outputs**, not adhering to a rigid methodology.

Balance prompt complexity against the value of results ⚖️. Elaborate prompts with multiple techniques might improve output quality by 10%, but cost three times as much in tokens and take longer to maintain. Sometimes simpler prompts that are "good enough" are the right choice, especially for high-volume applications where costs accumulate. Save your most sophisticated prompt engineering for tasks where quality improvements truly matter.

Finally, remember that **AI models continue evolving** 🚀. Today's models behave differently than those from six months ago, and tomorrow's models will be different still. Prompt engineering techniques that are essential now might become less important as models improve, or new techniques might emerge. Stay curious, keep experimenting, and be willing to adapt your approach as the technology evolves. The principles—**clarity, specificity, appropriate technique selection**—remain constant even as specific tactics change.

---

## 🎉 Congratulations!

You've completed this introduction to prompt engineering! You now understand:

✅ How **temperature** 🌡️ and **tokens** 🎫 affect model behavior and costs  
✅ Six tactics for writing clearer instructions 📝  
✅ How to ground responses in reference text 📚  
✅ When and how to use **few-shot prompting** 🎯  
✅ **Chain-of-thought** 🧮 for transparent reasoning  
✅ **Self-ask** 🔍 for complex question decomposition  
✅ How to choose and combine techniques effectively 🎨  

### 🚀 Next Steps

Practice is essential! Take these techniques and apply them to your own use cases. Start simple, experiment with different approaches, and build your intuition for what works when. Prompt engineering is a skill that develops through hands-on experience.

**Happy prompting!** 🎊✨